# FrameSight — Train YOLO11 on AutoDistill dataset (Google Colab)

1. On your PC: run `python scripts/autodistill_pipeline.py --zip`
2. Upload `data/autodistill/framesight_dataset.zip` below (or copy to Google Drive)
3. Run all cells → download `best.pt` → place in `weights/best.pt` on Windows

In [ ]:
!nvidia-smi
!pip install -q ultralytics>=8.3.0 pyyaml

In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

DATASET_ZIP = Path("framesight_dataset.zip")
DATASET_DIR = Path("/content/dataset")

if not DATASET_ZIP.exists():
    print("Upload framesight_dataset.zip from your PC pipeline (--zip)")
    uploaded = files.upload()
    DATASET_ZIP = Path(list(uploaded.keys())[0])

if DATASET_DIR.exists():
    import shutil
    shutil.rmtree(DATASET_DIR)

with zipfile.ZipFile(DATASET_ZIP, "r") as zf:
    zf.extractall("/content")

# Zip root may be 'framesight_dataset' or 'dataset'
for candidate in [Path("/content/framesight_dataset"), Path("/content/dataset"), DATASET_DIR]:
    if (candidate / "data.yaml").exists():
        DATASET_DIR = candidate
        break
else:
    raise FileNotFoundError("data.yaml not found in zip — re-run pipeline with --zip")

print("Dataset:", DATASET_DIR)
print("data.yaml:", (DATASET_DIR / "data.yaml").read_text()[:500])

In [ ]:
import yaml
from pathlib import Path
from ultralytics import YOLO

data_yaml = DATASET_DIR / "data.yaml"
with data_yaml.open() as f:
    cfg = yaml.safe_load(f)
cfg["path"] = str(DATASET_DIR.resolve())
with data_yaml.open("w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

EPOCHS = 100
BATCH = 16
IMGSZ = 640
PATIENCE = 20

model = YOLO("yolo11n.pt")
results = model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    device=0,
    project="/content/runs",
    name="framesight",
    exist_ok=True,
)

best_pt = Path(results.save_dir) / "weights" / "best.pt"
print("Best weights:", best_pt)

In [ ]:
from google.colab import files
files.download(str(best_pt))